# 01 · Dataset Exploration — BIRD & SynSQL-2.5M

Goal: understand the data **before** training — size, difficulty mix, SQL
complexity, schema breadth — so the fine-tuning recipe is grounded in
evidence rather than guesswork.

Runs on CPU. If the official BIRD files aren't present it falls back to the
bundled synthetic `data/sample/` so the notebook always executes.

In [ ]:
# --- SETUP: run me first (works on Colab AND locally) ---
import os, sys, subprocess
if not os.path.exists('src') and os.path.basename(os.getcwd()) != 'text2sql-finetuning':
    if not os.path.isdir('text2sql-finetuning'):
        subprocess.run(['git', 'clone', 'https://github.com/Shiverion/text2sql-finetuning.git'], check=True)
    os.chdir('text2sql-finetuning')
sys.path.insert(0, os.getcwd())
print('working dir :', os.getcwd())
print('src present :', os.path.isdir('src'))

In [ ]:
!pip install -q datasets pandas matplotlib

In [ ]:
import os, json, glob, re
import pandas as pd
import matplotlib.pyplot as plt
FIG = 'report/figures'; os.makedirs(FIG, exist_ok=True)

## 1. Load BIRD train (or fall back to the sample)

Official BIRD download (run on Colab if you want the full set):
```bash
!wget -q https://bird-bench.oss-cn-beijing.aliyuncs.com/train.zip && unzip -q train.zip
!wget -q https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip && unzip -q dev.zip
```
Links occasionally move — see https://bird-bench.github.io for the current ones,
or use a 🤗 mirror such as `lamini/bird_text_to_sql`.

In [ ]:
def load_examples():
    for p in ['train/train.json', 'dev/dev.json', 'data/train.json']:
        if os.path.exists(p):
            print('using', p)
            return json.load(open(p, encoding='utf-8')), p
    print('BIRD not found - using bundled sample')
    return json.load(open('data/sample/examples.json', encoding='utf-8')), 'sample'

rows, source = load_examples()
df = pd.DataFrame(rows)
df['sql'] = df.get('SQL', df.get('sql'))
print('examples:', len(df))
df.head(3)

## 2. Difficulty distribution
BIRD labels each dev question simple / moderate / challenging. The mix tells
us where accuracy will be hardest and how to read per-bucket EX later.

In [ ]:
if 'difficulty' in df and df['difficulty'].notna().any():
    vc = df['difficulty'].value_counts()
    print(vc)
    ax = vc.plot(kind='bar', title='Question difficulty', rot=0)
    ax.figure.tight_layout(); ax.figure.savefig(f'{FIG}/difficulty.png', dpi=120)
else:
    print('no difficulty labels in this split')

## 3. SQL complexity — length & keyword frequency
Proxies for how hard generation is: long queries, JOINs, aggregation, nesting.

In [ ]:
df['sql_len'] = df['sql'].str.len()
df['n_tokens'] = df['sql'].str.split().apply(len)
print(df[['sql_len','n_tokens']].describe())
ax = df['n_tokens'].plot(kind='hist', bins=30, title='SQL length (tokens)')
ax.figure.tight_layout(); ax.figure.savefig(f'{FIG}/sql_len.png', dpi=120)

In [ ]:
KEYWORDS = ['JOIN','LEFT JOIN','GROUP BY','ORDER BY','WHERE','HAVING',
            'DISTINCT','LIMIT','COUNT','SUM','AVG','MAX','MIN','CASE',
            'SELECT.*SELECT']  # last = nested subquery (regex)
up = df['sql'].str.upper()
freq = {k: int(up.str.contains(k, regex=True).sum()) for k in KEYWORDS}
freq = dict(sorted(freq.items(), key=lambda x: -x[1]))
print(json.dumps(freq, indent=2))
ax = pd.Series(freq).plot(kind='barh', title='SQL keyword frequency')
ax.invert_yaxis(); ax.figure.tight_layout(); ax.figure.savefig(f'{FIG}/keywords.png', dpi=120)

## 4. Schema breadth
How many distinct databases, and how wide are they? Wide schemas are the
main reason a small model fails: the relevant columns get lost in a long
prompt. This motivates schema serialization choices in `src/schema_utils.py`.

In [ ]:
if 'db_id' in df:
    print('distinct databases:', df['db_id'].nunique())
    print(df['db_id'].value_counts().head(10))

In [ ]:
# Column counts per DB. Sample DB by default; on full BIRD point the glob
# at dev_databases/**/*.sqlite.
import sqlite3
dbs = glob.glob('data/sample/db/**/*.sqlite', recursive=True) or glob.glob('dev_databases/**/*.sqlite', recursive=True)
for db in dbs[:10]:
    con = sqlite3.connect(db)
    tabs = [r[0] for r in con.execute("select name from sqlite_master where type='table'")]
    ncols = sum(len(con.execute(f'PRAGMA table_info("{t}")').fetchall()) for t in tabs)
    print(os.path.basename(db), '->', len(tabs), 'tables,', ncols, 'columns')
    con.close()

## 5. Takeaways → recipe

- **Schema is the prompt's bulk** → serialize compactly; keep `max_seq_length`
  large enough (2048) to fit the biggest schemas without truncation.
- **Evidence/hints matter** (BIRD) → always include them in the prompt.
- **JOIN + GROUP BY dominate** → these are the patterns LoRA must learn; the
  fintech use-case ('top merchants last quarter') is exactly this shape.
- **Difficulty is skewed to simple/moderate** → expect most EX gains there;
  challenging (nested, multi-join) will lag — call this out in the report.